# NHL Dashboard — Database Explorer

Use this notebook to explore the live SQLite database and validate data flowing through the pipeline
and diagnose dashboard issues without writing ad-hoc SQL scripts.

## Setup

```bash
pip install jupyter pandas matplotlib
jupyter notebook nhl-dashboard/notebooks/db_explorer.ipynb
```

Run Section 1 first to initialise the database path; then jump to any section.
Each section is self-contained — jump directly to the relevant one.

## Section 1 — Connection & Setup

Connects to `nhl-dashboard/backend/instance/nhl.db` and prints a quick health summary.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timezone, timedelta

from pathlib import Path

# Locate nhl.db regardless of the directory Jupyter was launched from.
# Walks upward from cwd until the repo layout (nhl-dashboard/ or backend/) is found.
_cwd = Path.cwd().resolve()
_db_path = None
for _base in (_cwd, *_cwd.parents):
    _candidate = _base / "nhl-dashboard" / "backend" / "instance" / "nhl.db"
    if _candidate.parent.exists():
        _db_path = _candidate
        break
    _candidate = _base / "backend" / "instance" / "nhl.db"
    if _candidate.parent.exists():
        _db_path = _candidate
        break
DB_PATH = _db_path or (_cwd / ".." / "backend" / "instance" / "nhl.db").resolve()


conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

version = conn.execute("SELECT sqlite_version()").fetchone()[0]
print(f"SQLite version: {version}")

EXPECTED_TABLES = {
    "team", "game", "odds_snapshot", "model_fair",
    "nhl_odds_partner", "nhl_odds_line", "nhl_historical_game",
}
rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
found_tables = {row[0] for row in rows}
print(f"\nFound tables: {sorted(found_tables)}")
missing = EXPECTED_TABLES - found_tables
if missing:
    print(f"WARNING — missing tables: {missing}")
else:
    print("All expected tables present ✓")

print("\n--- Row counts ---")
for table in sorted(EXPECTED_TABLES):
    if table in found_tables:
        n = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
        print(f"  {table}: {n}")
    else:
        print(f"  {table}: TABLE MISSING")


## Section 2 — `team` Table

The `team` table is populated from two sources:

1. **Startup seed** (Issue #112): On first run, the app calls the NHL Stats API and seeds all known franchises with full metadata (`team_id`, `franchise_id`, `full_name`, `league_id`, `raw_tricode`).
2. **Auto-append during slate refresh** (Issue #113): When a game references a tri-code not yet in the table (e.g. all-star games, expansion placeholders), a minimal row is inserted with only `tri_code` and `name` populated. These rows have `NULL` for `team_id` and other NHL Stats API fields.

**Why NULLs occur:** The auto-append path does not call the NHL Stats API — it inserts the bare minimum to satisfy the foreign key. A `NULL team_id` indicates the team has not yet been backfilled from the Stats API.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

teams = pd.read_sql_query(
    "SELECT tri_code, team_id, franchise_id, full_name, league_id, raw_tricode"
    " FROM team ORDER BY full_name",
    conn,
)
print(f"Total teams: {len(teams)}")
display(teams)

# Teams with NULL team_id were auto-appended by the slate refresh (Issue #113).
# They have not been backfilled from the NHL Stats API.
null_rows = teams[teams["team_id"].isna()]
if null_rows.empty:
    print("No teams with NULL team_id — all rows seeded from NHL Stats API ✓")
else:
    print(f"WARNING — {len(null_rows)} team(s) with NULL team_id (auto-appended, not yet seeded):")
    display(null_rows[["tri_code", "full_name", "team_id"]])

### Section 2b — Team-to-Game Join

Resolves `away_code` / `home_code` FK references in the `game` table to full team names via `tri_code`.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

joined = pd.read_sql_query(
    """
    SELECT g.game_id, t_away.full_name AS away, t_home.full_name AS home
    FROM game g
    JOIN team t_away ON t_away.tri_code = g.away_code
    JOIN team t_home ON t_home.tri_code = g.home_code
    """,
    conn,
)
print(f"Games with resolved team names: {len(joined)}")
display(joined)

## Section 3 — `game` Table

Shows the last 10 games ordered by `start_est` descending, status distribution, stale-poll flag, live-game integrity, and FK orphan check.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timezone, timedelta

conn = sqlite3.connect(DB_PATH)

games = pd.read_sql_query(
    """
    SELECT game_id, away_code, home_code, start_est, status, period, clock,
           away_score, home_score, updated_at
    FROM game
    ORDER BY start_est DESC LIMIT 10
    """,
    conn,
)
print(f"Last 10 games: {len(games)}")
display(games)

print("\n--- Status distribution ---")
print(games["status"].value_counts().to_string())

# Flag stale updated_at (> 10 minutes old)
now_utc = datetime.now(timezone.utc)
stale_threshold = (now_utc - timedelta(minutes=10)).replace(tzinfo=None)

def _parse_utc(ts):
    if ts is None:
        return None
    try:
        dt = datetime.fromisoformat(ts.replace("Z", "+00:00"))
        return dt.replace(tzinfo=None)
    except Exception:
        return None

games["_updated_dt"] = games["updated_at"].apply(_parse_utc)
stale = games[games["_updated_dt"].notna() & (games["_updated_dt"] < stale_threshold)]
if stale.empty:
    print("\nNo stale games (updated_at > 10 min old) ✓")
else:
    print(f"\nWARNING — {len(stale)} game(s) with stale updated_at:")
    display(stale[["game_id", "away_code", "home_code", "status", "updated_at"]])

# Flag live games missing period or clock
live_incomplete = games[
    (games["status"] == "live") & (games["period"].isna() | games["clock"].isna())
]
if live_incomplete.empty:
    print("\nNo live games with missing period/clock ✓")
else:
    print(f"\nWARNING — {len(live_incomplete)} live game(s) with NULL period or clock:")
    display(live_incomplete[["game_id", "away_code", "home_code", "period", "clock"]])

# FK integrity: away_code and home_code must exist in team table
# Uses tri_code (PK after Issue #111 schema migration)
team_codes = pd.read_sql_query("SELECT tri_code FROM team", conn)["tri_code"].tolist()
orphan_away = games[~games["away_code"].isin(team_codes)]
orphan_home = games[~games["home_code"].isin(team_codes)]
if orphan_away.empty and orphan_home.empty:
    print("\nNo FK orphans in game table ✓")
else:
    if not orphan_away.empty:
        print(f"\nWARNING — {len(orphan_away)} game(s) with unknown away_code:")
        display(orphan_away[["game_id", "away_code", "home_code"]])
    if not orphan_home.empty:
        print(f"\nWARNING — {len(orphan_home)} game(s) with unknown home_code:")
        display(orphan_home[["game_id", "away_code", "home_code"]])

## Section 3.5 — NHL API Odds

Data source: **NHL API `/v1/score/now`** — populated by the score refresh job (APScheduler).

**Availability window:** These tables are populated on game days only, after the scheduled score refresh job has fired at least once. On non-game days, `nhl_odds_line` will be empty.

Two tables:
- `nhl_odds_partner` — betting partner registry (static, seeded from the `oddsPartners` array)
- `nhl_odds_line` — per-game, per-partner moneyline snapshots, one row per 3-minute poll cycle

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

partners = pd.read_sql_query(
    "SELECT partner_id, name, country, image_url, site_url,"
    " bg_color, text_color, accent_color"
    " FROM nhl_odds_partner ORDER BY partner_id",
    conn,
)
if partners.empty:
    print("No nhl_odds_partner rows yet — run on a game day after the score refresh job has fired.")
else:
    print(f"Total partners: {len(partners)}")
    display(partners)

### Odds Format Reference

The NHL API serves odds in two formats depending on the partner’s market:

| Format | Description | Example |
|--------|-------------|-------|
| **American** | Negative = favourite, positive = underdog | `-152` (favourite) · `+126` (underdog) |
| **Decimal** | Multiplier including stake; `decimal ≈ (100 / abs(american)) + 1` when negative | `1.67` ≈ `-152` American |

**Partner format by ID (confirmed from NHL API):**

| Partner ID | Name | Format |
|-----------|------|--------|
| 7 | FanDuel | American |
| 8 | BetMGM | American |
| 9 | DraftKings | American |
| 3 | Unibet | Decimal |
| 6 | Betway | Decimal |
| 10 | 888sport | Decimal |

North American partners (IDs 7, 8, 9) use American format. European partners (IDs 3, 6, 10) use decimal format.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

lines = pd.read_sql_query(
    """
    SELECT l.game_id, p.name AS partner_name, l.fetched_at, l.away_value, l.home_value
    FROM nhl_odds_line l
    JOIN nhl_odds_partner p ON p.partner_id = l.partner_id
    ORDER BY l.fetched_at DESC
    LIMIT 100
    """,
    conn,
)
if lines.empty:
    print("No nhl_odds_line rows yet — run on a game day after the score refresh job has fired.")
else:
    print(f"Recent odds lines (up to 100): {len(lines)}")
    display(lines)

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

latest = pd.read_sql_query(
    """
    SELECT l.game_id, p.name AS partner_name, l.away_value, l.home_value, l.fetched_at
    FROM nhl_odds_line l
    JOIN nhl_odds_partner p ON p.partner_id = l.partner_id
    INNER JOIN (
        SELECT game_id, partner_id, MAX(fetched_at) AS max_fetched
        FROM nhl_odds_line
        GROUP BY game_id, partner_id
    ) lk ON l.game_id = lk.game_id
        AND l.partner_id = lk.partner_id
        AND l.fetched_at = lk.max_fetched
    ORDER BY l.game_id, p.name
    """,
    conn,
)
if latest.empty:
    print("No nhl_odds_line rows yet — run on a game day after the score refresh job has fired.")
else:
    print(f"Latest odds per game+partner: {len(latest)} row(s)")
    display(latest)

### Cross-Source Comparison: NHL API vs. `odds_snapshot`

The table below joins `nhl_odds_line` (sourced from the NHL API `/v1/score/now` odds arrays) with the most recent `odds_snapshot` row for the same `game_id` (sourced from the-odds-api).

- **`nhl_away` / `nhl_home`**: String values from the NHL API — may be American (`-152`) or decimal (`1.67`) format
- **`snap_away_ml` / `snap_home_ml`**: Integer American odds from the-odds-api

A `NULL` in the `snap_*` columns means no `odds_snapshot` exists yet for that game.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

comparison = pd.read_sql_query(
    """
    SELECT g.game_id, g.away_code, g.home_code,
           p.name AS partner_name,
           l.away_value AS nhl_away, l.home_value AS nhl_home,
           l.fetched_at AS nhl_fetched_at,
           o.away_ml AS snap_away_ml, o.home_ml AS snap_home_ml,
           o.fetched_at AS snap_fetched_at
    FROM game g
    JOIN nhl_odds_line l ON l.game_id = g.game_id
    JOIN nhl_odds_partner p ON p.partner_id = l.partner_id
    LEFT JOIN (
        SELECT game_id, away_ml, home_ml, fetched_at
        FROM odds_snapshot
        WHERE fetched_at IN (
            SELECT MAX(fetched_at) FROM odds_snapshot GROUP BY game_id
        )
    ) o ON o.game_id = g.game_id
    ORDER BY g.game_id, p.name
    """,
    conn,
)
if comparison.empty:
    print("No matching game_id in both nhl_odds_line and odds_snapshot — run on a game day.")
else:
    print(f"Cross-source comparison: {len(comparison)} row(s)")
    display(comparison)

## Section 4 — `odds_snapshot` Table

Shows snapshot counts, most recent odds, zero-snapshot flag, implied probability scale check (see Issue #106), and a sparkline for a selected game.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt

conn = sqlite3.connect(DB_PATH)

# Snapshot count per game
counts = pd.read_sql_query(
    "SELECT game_id, COUNT(*) AS snapshot_count FROM odds_snapshot GROUP BY game_id",
    conn
)
print("Snapshot counts per game:")
display(counts)

# Most recent snapshot for each game
latest = pd.read_sql_query(
    """
    SELECT o.game_id, o.fetched_at, o.book,
           o.away_ml, o.home_ml, o.away_implied, o.home_implied
    FROM odds_snapshot o
    INNER JOIN (
        SELECT game_id, MAX(fetched_at) AS max_fetched
        FROM odds_snapshot GROUP BY game_id
    ) lk ON o.game_id = lk.game_id AND o.fetched_at = lk.max_fetched
    """,
    conn
)
print("\nMost recent snapshot per game:")
display(latest)

# Flag games with zero snapshots
today = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d")
today_game_ids = [
    r[0] for r in conn.execute(
        "SELECT game_id FROM game WHERE start_utc LIKE ?", (f"{today}%",)
    ).fetchall()
]
snapped_ids = set(counts["game_id"].tolist())
no_snaps = [gid for gid in today_game_ids if gid not in snapped_ids]
if no_snaps:
    print(f"\nWARNING — {len(no_snaps)} today game(s) with zero snapshots: {no_snaps}")
else:
    print("\nAll today games have at least one snapshot ✓")

# Scale check — Issue #106: implied probs should be 0-100 percentage points after fix.
# Sum of away_implied + home_implied should be ~100-115 with vig.
# A sum < 5 suggests the old 0-1 scale is still in use.
if not latest.empty:
    latest["implied_sum"] = latest["away_implied"] + latest["home_implied"]
    scale_issues = latest[(latest["implied_sum"] < 5) | (latest["implied_sum"] > 150)]
    if scale_issues.empty:
        print("\nImplied probability scale looks correct (0-100 range) ✓")
    else:
        print("\nWARNING — scale anomaly detected (see Issue #106 for expected 0-100 range):")
        display(scale_issues[["game_id", "away_implied", "home_implied", "implied_sum"]])

# Sparkline: away_implied over time for a user-selected game
# Change SELECTED_GAME_ID to the game you want to inspect
SELECTED_GAME_ID = today_game_ids[0] if today_game_ids else None

if SELECTED_GAME_ID is not None:
    series = pd.read_sql_query(
        "SELECT fetched_at, away_implied FROM odds_snapshot WHERE game_id = ? ORDER BY fetched_at",
        conn, params=[SELECTED_GAME_ID]
    )
    if not series.empty:
        fig, ax = plt.subplots(figsize=(8, 3))
        ax.plot(series["fetched_at"], series["away_implied"], marker="o", linewidth=1.5)
        ax.set_title(f"Away implied probability over time — game {SELECTED_GAME_ID}")
        ax.set_xlabel("fetched_at")
        ax.set_ylabel("away_implied (%)")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()
    else:
        print(f"No snapshots for game {SELECTED_GAME_ID} yet.")
else:
    print("No games today — sparkline skipped.")


## Section 5 — `model_fair` Table

Shows fair-value rows alongside recent implied odds, calculates edge, and flags missing rows and sum-to-1 violations.

In [ ]:
import sqlite3
import pandas as pd
import datetime as dt

conn = sqlite3.connect(DB_PATH)

today = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d")

# Join model_fair with latest implied odds for today's games
combined = pd.read_sql_query(
    """
    SELECT g.game_id, g.away_code, g.home_code,
           mf.away_fair, mf.home_fair, mf.computed_at,
           o.away_implied, o.home_implied, o.fetched_at
    FROM game g
    LEFT JOIN model_fair mf ON mf.game_id = g.game_id
    LEFT JOIN (
        SELECT game_id, away_implied, home_implied, fetched_at
        FROM odds_snapshot
        WHERE fetched_at IN (
            SELECT MAX(fetched_at) FROM odds_snapshot GROUP BY game_id
        )
    ) o ON o.game_id = g.game_id
    WHERE g.start_utc LIKE :today
    ORDER BY g.start_utc
    """,
    conn, params={"today": f"{today}%"}
)
display(combined)

# Edge: fair.away − implied.away (both should be on the same scale)
if not combined.empty and combined["away_fair"].notna().any():
    combined["edge_away"] = combined["away_fair"] - combined["away_implied"]
    combined["edge_home"] = combined["home_fair"] - combined["home_implied"]
    print("\nEdge per game (fair − implied):")
    display(combined[["game_id", "away_code", "home_code", "edge_away", "edge_home"]])

# Flag games with no model_fair row
no_fair = combined[combined["away_fair"].isna()]
if no_fair.empty:
    print("\nAll today games have a model_fair row ✓")
else:
    print(f"\nWARNING — {len(no_fair)} game(s) with no model_fair row (compute job may not have run):")
    display(no_fair[["game_id", "away_code", "home_code"]])

# Sum-to-1 check: away_fair + home_fair should be ~1.0 or ~100 depending on scale
fair_rows = combined[combined["away_fair"].notna()].copy()
if not fair_rows.empty:
    fair_rows["fair_sum"] = fair_rows["away_fair"] + fair_rows["home_fair"]
    bad = fair_rows[
        ~(((fair_rows["fair_sum"] - 1.0).abs() < 0.02) |
          ((fair_rows["fair_sum"] - 100.0).abs() < 2.0))
    ]
    if bad.empty:
        print("\nmodel_fair probabilities sum to ~1 or ~100 ✓")
    else:
        print(f"\nWARNING — {len(bad)} model_fair row(s) with unexpected sum:")
        display(bad[["game_id", "away_fair", "home_fair", "fair_sum"]])


## Section 6 — End-to-End Pipeline Check

Runs all validation checks and prints a pass/fail health report.

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timezone, timedelta

conn = sqlite3.connect(DB_PATH)

today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
EXPECTED_TABLES = {
    "team", "game", "odds_snapshot", "model_fair",
    "nhl_odds_partner", "nhl_odds_line", "nhl_historical_game",
}

def _check(label, passed):
    status = "✓" if passed else "✗"
    print(f"  {status}  {label}")
    return passed

print("=== Pipeline Health Report ===")
print(f"Generated: {datetime.now(timezone.utc).isoformat()}")
print()

# 1 — all expected tables exist
rows = conn.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
found = {r[0] for r in rows}
_check("All expected tables exist", EXPECTED_TABLES <= found)

# 2 — at least one game for today
today_count = conn.execute(
    "SELECT COUNT(*) FROM game WHERE start_utc LIKE ?", (f"{today}%",)
).fetchone()[0]
_check(f"At least one game for today ({today})", today_count > 0)

# 3 — no stale updated_at (> 10 min)
stale_cutoff = (datetime.now(timezone.utc) - timedelta(minutes=10)).isoformat()
stale_count = conn.execute(
    "SELECT COUNT(*) FROM game WHERE start_utc LIKE ? AND updated_at < ?",
    (f"{today}%", stale_cutoff)
).fetchone()[0]
_check("No games with stale updated_at (> 10 min)", stale_count == 0)

# 4 — all today's games have at least one snapshot
today_ids = [r[0] for r in conn.execute(
    "SELECT game_id FROM game WHERE start_utc LIKE ?", (f"{today}%",)
).fetchall()]
if today_ids:
    placeholders = ",".join("?" * len(today_ids))
    snapped = conn.execute(
        f"SELECT COUNT(DISTINCT game_id) FROM odds_snapshot WHERE game_id IN ({placeholders})",
        today_ids
    ).fetchone()[0]
    _check("All games have at least one odds snapshot", snapped == len(today_ids))
else:
    _check("All games have at least one odds snapshot", True)

# 5 — all today's games have a model_fair row
if today_ids:
    fair_count = conn.execute(
        f"SELECT COUNT(*) FROM model_fair WHERE game_id IN ({placeholders})",
        today_ids
    ).fetchone()[0]
    _check("All games have a model_fair row", fair_count == len(today_ids))
else:
    _check("All games have a model_fair row", True)

# 6 — no FK orphans in odds_snapshot
orphans = conn.execute(
    "SELECT COUNT(*) FROM odds_snapshot WHERE game_id NOT IN (SELECT game_id FROM game)"
).fetchone()[0]
_check("No FK orphans in odds_snapshot", orphans == 0)

# 7 — implied probability scale consistent (Issue #106)
scale_ok = True
if today_ids:
    sample = conn.execute(
        f"SELECT away_implied, home_implied FROM odds_snapshot WHERE game_id IN ({placeholders}) LIMIT 20",
        today_ids
    ).fetchall()
    tiny = [r for r in sample if r[0] is not None and (r[0] + r[1]) < 5]
    scale_ok = len(tiny) == 0 or len(sample) == 0
_check("Implied probability scale consistent (see Issue #106)", scale_ok)

# 8 — nhl_historical_game has rows (backfill has run)
hist_count = conn.execute("SELECT COUNT(*) FROM nhl_historical_game").fetchone()[0]
_check(f"nhl_historical_game backfill has run ({hist_count:,} rows)", hist_count > 0)

print()
print("Done.")


## Section 7 — `nhl_historical_game` Table

Historical game records backfilled from the NHL Stats REST API
(`GET https://api.nhle.com/stats/rest/en/game`). Populated by
`ingest_historical_games()` in `services/historical.py` (Issue #121).

**Source:** `https://api.nhle.com/stats/rest/en/game` — full historical set.  
**Update strategy:** Upsert by `game_id` — safe to re-run at any time.  
**Team identity:** Uses integer `home_team_id` / `visiting_team_id` (matches `team.team_id`),
not `tri_code` FK references.

Sub-sections:
- **7a** — Row count, season distribution, and most-recent rows
- **7b** — Filter by season
- **7c** — Game type breakdown (regular season vs. playoffs)
- **7d** — Join with `team` table to resolve team names


### 7a — Row Count & Season Distribution

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

total = conn.execute("SELECT COUNT(*) FROM nhl_historical_game").fetchone()[0]
print(f"Total rows in nhl_historical_game: {total:,}")

if total == 0:
    print("Table is empty — run ingest_historical_games() to backfill.")
    print("Example (from Flask shell or script):")
    print("  from services.historical import ingest_historical_games")
    print("  ingest_historical_games()")
else:
    # Season distribution
    seasons = pd.read_sql_query(
        """
        SELECT season, COUNT(*) AS game_count,
               SUM(CASE WHEN game_type = 2 THEN 1 ELSE 0 END) AS regular,
               SUM(CASE WHEN game_type = 3 THEN 1 ELSE 0 END) AS playoffs
        FROM nhl_historical_game
        GROUP BY season
        ORDER BY season DESC
        LIMIT 10
        """,
        conn,
    )
    print("\n--- Last 10 seasons (most recent first) ---")
    display(seasons)

    # 10 most recent game rows
    recent = pd.read_sql_query(
        """
        SELECT game_id, game_date, season, game_type,
               home_team_id, visiting_team_id, home_score, visiting_score, period
        FROM nhl_historical_game
        ORDER BY game_date DESC, game_id DESC
        LIMIT 10
        """,
        conn,
    )
    print("\n--- 10 most recent game rows ---")
    display(recent)


### 7b — Filter by Season

Change `SELECTED_SEASON` to inspect any season (e.g. `20252026` = 2025-26 season).

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

SELECTED_SEASON = 20252026  # change to inspect another season

games = pd.read_sql_query(
    """
    SELECT game_id, game_date, game_type, game_number,
           home_team_id, visiting_team_id,
           home_score, visiting_score, period, game_state_id
    FROM nhl_historical_game
    WHERE season = :season
    ORDER BY game_date, game_number
    """,
    conn, params={"season": SELECTED_SEASON},
)

if games.empty:
    print(f"No rows for season {SELECTED_SEASON} — backfill may not have run yet.")
else:
    print(f"Season {SELECTED_SEASON}: {len(games):,} games")
    display(games.head(20))


### 7c — Game Type Breakdown

`game_type` codes: `1` = preseason, `2` = regular season, `3` = playoffs, `4` = all-star.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

breakdown = pd.read_sql_query(
    """
    SELECT
        game_type,
        CASE game_type
            WHEN 1 THEN 'Preseason'
            WHEN 2 THEN 'Regular Season'
            WHEN 3 THEN 'Playoffs'
            WHEN 4 THEN 'All-Star'
            ELSE 'Other'
        END AS type_label,
        COUNT(*) AS game_count
    FROM nhl_historical_game
    GROUP BY game_type
    ORDER BY game_type
    """,
    conn,
)

if breakdown.empty:
    print("No rows — backfill may not have run yet.")
else:
    print("Game type breakdown (all-time):")
    display(breakdown)


### 7d — Join with `team` Table to Resolve Team Names

Joins `home_team_id` / `visiting_team_id` to `team.team_id` to show readable team names.
Rows where `team_id` is not in the `team` table appear as NULL (historic franchises
not yet in the seed data).

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

SELECTED_SEASON = 20252026  # change to inspect another season

resolved = pd.read_sql_query(
    """
    SELECT
        h.game_id,
        h.game_date,
        h.season,
        h.game_type,
        t_vis.full_name  AS visiting_team,
        h.visiting_score AS vis_score,
        h.home_score     AS home_score,
        t_home.full_name AS home_team,
        h.period,
        h.game_state_id
    FROM nhl_historical_game h
    LEFT JOIN team t_home ON t_home.team_id = h.home_team_id
    LEFT JOIN team t_vis  ON t_vis.team_id  = h.visiting_team_id
    WHERE h.season = :season
    ORDER BY h.game_date DESC, h.game_id DESC
    LIMIT 20
    """,
    conn, params={"season": SELECTED_SEASON},
)

if resolved.empty:
    print(f"No rows for season {SELECTED_SEASON} — backfill may not have run yet.")
else:
    print(f"Season {SELECTED_SEASON} — 20 most recent games with resolved team names:")
    display(resolved)

    # Warn about unresolved team IDs
    unresolved = resolved[resolved['visiting_team'].isna() | resolved['home_team'].isna()]
    if not unresolved.empty:
        print(f"\nWARNING — {len(unresolved)} row(s) with unresolved team IDs:")
        display(unresolved[['game_id', 'visiting_team', 'home_team']])
    else:
        print("\nAll team IDs resolved to team names ✓")
